# NTFS Timestomping Detection Tool v3.0
# Step 3: Run Detection

---

## Overview

This notebook applies the trained Random Forest baseline model to detect timestamp manipulation attacks.

**Final Model Selected**: Random Forest Tuned

**Why this model?**
After comprehensive evaluation (Phase 5 hyperparameter tuning and threshold optimization), the Phase 4 baseline model was selected as the final production model because:
1. GridSearchCV hyperparameter tuning degraded performance (F1 dropped to 0.5950)
2. Threshold optimization confirmed the model is already well-calibrated at 0.5
3. The baseline achieves strong recall (92.31%) for attack detection
4. Low false positive rate (0.10%) makes manual review feasible

For detailed analysis, see `Phase 5 - V2 Hyperparameter Tuning/03_Final_Model_Comparison_v3.ipynb`

---

## Input

`forensic_features.csv` from Step 2

## Output

- `predictions.csv` - All events with predictions and confidence scores
- `flagged_files.csv` - Only suspected timestomped events (includes LSN/USN for traceability)



## 1. Setup

In [36]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib  
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully")


Libraries loaded successfully


## 2. Load Features and Model

In [37]:
print("=" * 80)
print("LOADING DATA AND MODEL")
print("=" * 80)

# Paths
DATA_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/12-KSK')
MODEL_PATH = Path('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/random_forest_tuned_v2.pkl')
OUTPUT_DIR = DATA_DIR

print(f"Loading model from: {MODEL_PATH}")
model = joblib.load(MODEL_PATH)
print(f"\n✓ Model loaded successfully")
print(f"  Model type: {type(model).__name__}")
print(f"  Expected features: {len(model.feature_names_in_)}")

# Load features
print("\n1. Loading forensic features...")
features_file = DATA_DIR / 'forensic_features.csv'

if not features_file.exists():
    raise FileNotFoundError(f"Features file not found: {features_file}\nPlease run 02_Feature_Engineering.ipynb first!")

# IMPORTANT: Force these columns to be read as strings (not auto-converted to bool)
# These need to stay as object type for one-hot encoding
dtype_dict = {
    'copied_from_file': str,
    'creation_time_changed_to_past': str,
    'modified_time_changed_to_past': str,
    'accessed_time_changed_to_past': str,
    'mft_modified_time_changed_to_past': str
}

df = pd.read_csv(features_file, low_memory=False, dtype=dtype_dict)
print(f"   Loaded {len(df):,} events")
print(f"   {len(df.columns)} columns")

# Load trained model
print("\n2. Loading trained model...")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

# Try loading with joblib (recommended for scikit-learn models)
try:
    model = joblib.load(MODEL_PATH)
    print(f"   Loaded Random Forest model (using joblib)")
except Exception as e:
    print(f"   Warning: joblib failed, trying pickle...")
    import pickle
    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)
    print(f"   Loaded Random Forest model (using pickle)")

print(f"   Model type: {type(model).__name__}")

print("\n" + "=" * 80)
print("Data and model loaded successfully")
print("=" * 80)

LOADING DATA AND MODEL
Loading model from: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/random_forest_tuned_v2.pkl

✓ Model loaded successfully
  Model type: RandomForestClassifier
  Expected features: 50

1. Loading forensic features...
   Loaded 17,418 events
   55 columns

2. Loading trained model...
   Loaded Random Forest model (using joblib)
   Model type: RandomForestClassifier

Data and model loaded successfully


## 3. Prepare Features for Prediction

Extract only the features used by the model (exclude metadata columns).

In [38]:
print("=" * 80)
print("PREPARING FEATURES")
print("=" * 80)

# Define metadata columns (not used as features)
metadata_cols = [
    'eventtime', 'eventtime_dt',
    'filename', 'filepath', 'merge_key',
    'lf_redo', 'lf_target_vcn',
    'usn_file_attribute', 'usn_file_ref_num', 'usn_parent_file_ref_num',
    'case_id'  # If it exists
]

# Get feature columns (everything except metadata)
feature_cols = [col for col in df.columns if col not in metadata_cols]

print(f"\n1. Extracting features...")
print(f"   Total columns: {len(df.columns)}")
print(f"   Metadata columns: {len(metadata_cols)}")
print(f"   Feature columns: {len(feature_cols)}")

# Extract features
X = df[feature_cols].copy()
print(f"   Initial feature matrix: {X.shape}")

# 2. Apply same preprocessing as training (PHASE 5 TUNED MODEL)
print(f"\n2. Applying training preprocessing (Phase 5 tuned model)...")

# First handle object columns for one-hot encoding
object_cols = X.select_dtypes(include=['object']).columns.tolist()

# Drop high-cardinality columns (>1000 unique values)
cols_to_drop = [col for col in object_cols if X[col].nunique() > 1000]
if cols_to_drop:
    print(f"   Dropping {len(cols_to_drop)} high-cardinality columns: {cols_to_drop}")
    X = X.drop(columns=cols_to_drop)

# One-hot encode low-cardinality object columns
cols_to_encode = [col for col in object_cols if col not in cols_to_drop and col in X.columns]

# Special handling for different column types
boolean_like_cols = ['copied_from_file', 'creation_time_changed_to_past', 'modified_time_changed_to_past', 
                     'accessed_time_changed_to_past', 'mft_modified_time_changed_to_past']
# Force label encoding for timestamp columns (they were label encoded during training)
timestamp_cols = ['lf_creation_time_before', 'lf_creation_time_after', 'lf_modified_time_before', 
                  'lf_modified_time_after', 'lf_accessed_time_before', 'lf_accessed_time_after',
                  'lf_mft_modified_time_before', 'lf_mft_modified_time_after', 'usn_event_info']

if cols_to_encode:
    print(f"   Encoding {len(cols_to_encode)} categorical columns...")
    from sklearn.preprocessing import LabelEncoder
    
    for col in cols_to_encode:
        # CRITICAL FIX: Match Phase 5 tuned model preprocessing
        # Boolean columns: fill NaN with False (not 'missing')
        if col in boolean_like_cols:
            X[col] = X[col].fillna(False)
            X[col] = X[col].astype(str)
        else:
            X[col] = X[col].fillna('missing')
        
        # Force label encoding for timestamp columns
        if col in timestamp_cols:
            unique_count = X[col].nunique()
            print(f"     Label encoding {col} ({unique_count} unique values, forced for timestamp)")
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col])
        # One-hot encode if < 10 unique values
        elif X[col].nunique() < 10:
            unique_count = X[col].nunique()
            
            # CRITICAL: Use drop_first=True for boolean columns (Phase 5 model)
            # This creates only _True columns (False is implicit baseline)
            if col in boolean_like_cols:
                print(f"     One-hot encoding {col} ({unique_count} unique values, drop_first=True)")
                dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
                print(f"       Created columns: {', '.join(dummies.columns.tolist())}")
            else:
                # Other categorical: use drop_first=False
                print(f"     One-hot encoding {col} ({unique_count} unique values, drop_first=False)")
                dummies = pd.get_dummies(X[col], prefix=col, drop_first=False)
                print(f"       Created columns: {', '.join(dummies.columns.tolist())}")
            
            X = pd.concat([X.drop(columns=[col]), dummies], axis=1)
        else:
            # Label encode if >= 10 unique values
            unique_count = X[col].nunique()
            print(f"     Label encoding {col} ({unique_count} unique values)")
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col])

# NOW convert remaining boolean columns to int (after one-hot encoding is done)
bool_cols = X.select_dtypes(include=['bool']).columns.tolist()
if bool_cols:
    print(f"   Converting {len(bool_cols)} boolean columns to int...")
    for col in bool_cols:
        X[col] = X[col].astype(int)

# Fill remaining missing values with -1
X = X.fillna(-1)

print(f"   Preprocessed feature matrix: {X.shape}")

# 3. Align features with model expectations
print(f"\n3. Aligning features with trained model...")

# Get model's expected feature names
model_features = model.feature_names_in_
print(f"   Model expects {len(model_features)} features")
print(f"   We have {len(X.columns)} features")

# Find missing and extra features
missing_features = set(model_features) - set(X.columns)
extra_features = set(X.columns) - set(model_features)

if missing_features:
    print(f"\n   WARNING: {len(missing_features)} missing features (filled with 0):")
    for feat in sorted(missing_features)[:10]:  # Show first 10
        print(f"      - {feat}")
    if len(missing_features) > 10:
        print(f"      ... and {len(missing_features) - 10} more")
    
    # This is expected for categorical variables not present in test data
    for feat in missing_features:
        X[feat] = 0

if extra_features:
    print(f"\n   Dropping {len(extra_features)} extra features:")
    for feat in sorted(extra_features)[:10]:  # Show first 10
        print(f"      - {feat}")
    if len(extra_features) > 10:
        print(f"      ... and {len(extra_features) - 10} more")
    X = X.drop(columns=list(extra_features))

# Reorder columns to match model's expected order
X = X[model_features]

print(f"\n   Final feature matrix: {X.shape}")
print(f"     Features match model expectations")

print("\n" + "=" * 80)
print("Features ready for prediction")
print("=" * 80)

PREPARING FEATURES

1. Extracting features...
   Total columns: 55
   Metadata columns: 11
   Feature columns: 49
   Initial feature matrix: (17418, 49)

2. Applying training preprocessing (Phase 5 tuned model)...
   Dropping 1 high-cardinality columns: ['usn_file_reference_number']
   Encoding 17 categorical columns...
     One-hot encoding lf_event (4 unique values, drop_first=False)
       Created columns: lf_event_CreationTime Update Event, lf_event_Time Reversal Event, lf_event_Time Reversal Event & Changing FileAttribute, lf_event_missing
     Label encoding usn_event_info (49 unique values, forced for timestamp)
     Label encoding usn_parent_file_reference_number (269 unique values)
     One-hot encoding source (3 unique values, drop_first=False)
       Created columns: source_both, source_logfile_only, source_usnjrnl_only
     Label encoding lf_creation_time_before (83 unique values, forced for timestamp)
     Label encoding lf_creation_time_after (21 unique values, forced for

## 4. Run Predictions
Apply the trained model to detect timestamp manipulation

In [39]:
print("=" * 80)
print("RUNNING PREDICTIONS")
print("=" * 80)

print(f"\nAnalyzing {len(X):,} events...")

# Get predictions (0 = benign, 1 = timestomped)
predictions = model.predict(X)

# Get probability scores (confidence)
probabilities = model.predict_proba(X)
confidence_scores = probabilities[:, 1]  # Probability of being timestomped

# Add predictions to dataframe
df['predicted_timestomped'] = predictions
df['confidence_score'] = confidence_scores

# Add confidence level interpretation
def get_confidence_level(score):
    if score >= 0.70:
        return 'HIGH'
    elif score >= 0.50:
        return 'MEDIUM'
    else:
        return 'LOW'

df['confidence_level'] = df['confidence_score'].apply(get_confidence_level)

# Classification
df['classification'] = df['predicted_timestomped'].map({
    0: 'Benign',
    1: 'Timestomped'
})

# Summary
timestomped_count = (predictions == 1).sum()
benign_count = (predictions == 0).sum()

print("\n" + "=" * 80)
print("DETECTION RESULTS")
print("=" * 80)

print(f"\nTotal events analyzed: {len(df):,}")
print(f"\nClassification:")
print(f"  Benign events: {benign_count:,} ({benign_count/len(df)*100:.1f}%)")
print(f"  Timestomped events: {timestomped_count:,} ({timestomped_count/len(df)*100:.1f}%)")

# Show confidence score distribution
print(f"\nConfidence Score Distribution:")
print(f"  HIGH (>=0.70):   {(df['confidence_level'] == 'HIGH').sum():,} events")
print(f"  MEDIUM (0.50-0.69): {(df['confidence_level'] == 'MEDIUM').sum():,} events")
print(f"  LOW (<0.50):    {(df['confidence_level'] == 'LOW').sum():,} events")

print(f"\nConfidence Score Statistics:")
print(f"  Maximum: {df['confidence_score'].max():.4f}")
print(f"  Mean:    {df['confidence_score'].mean():.4f}")
print(f"  Median:  {df['confidence_score'].median():.4f}")
print(f"  Minimum: {df['confidence_score'].min():.4f}")

if timestomped_count > 0:
    print(f"\nFlagged Events (Timestomped):")
    flagged = df[df['predicted_timestomped'] == 1]
    print(f"  Average confidence: {flagged['confidence_score'].mean():.1%}")
    print(f"  Minimum confidence: {flagged['confidence_score'].min():.1%}")
    print(f"  Maximum confidence: {flagged['confidence_score'].max():.1%}")
    
    # Breakdown by confidence level
    print(f"\n  By confidence level:")
    for level in ['HIGH', 'MEDIUM', 'LOW']:
        count = (flagged['confidence_level'] == level).sum()
        if count > 0:
            print(f"    {level}: {count:,} events")

print("\n" + "=" * 80)
print("Prediction complete")
print("=" * 80)


RUNNING PREDICTIONS

Analyzing 17,418 events...

DETECTION RESULTS

Total events analyzed: 17,418

Classification:
  Benign events: 17,417 (100.0%)
  Timestomped events: 1 (0.0%)

Confidence Score Distribution:
  HIGH (>=0.70):   0 events
  MEDIUM (0.50-0.69): 1 events
  LOW (<0.50):    17,417 events

Confidence Score Statistics:
  Maximum: 0.5088
  Mean:    0.0578
  Median:  0.0499
  Minimum: 0.0299

Flagged Events (Timestomped):
  Average confidence: 50.9%
  Minimum confidence: 50.9%
  Maximum confidence: 50.9%

  By confidence level:
    MEDIUM: 1 events

Prediction complete


## 5. Top 10 High-Confidence Predictions

Display the events with highest confidence scores for immediate triage.

In [40]:
print("=" * 80)
print("TOP 10 HIGHEST CONFIDENCE PREDICTIONS")
print("=" * 80)

# Get top 10 by confidence score
top_10 = df.nlargest(10, 'confidence_score')[[
    'eventtime', 'lf_lsn', 'usn_usn',  'filename', 'filepath',
    'confidence_score', 'predicted_timestomped', 'classification',
    'source', 'has_logfile_evidence', 'has_usnjrnl_evidence',
    'cross_artifact_validation_score',
    'timestamp_manipulation_pattern_score'
]].copy()

print(f"\nShowing top 10 events by confidence score:")
print(f"(These are the most suspicious events regardless of classification threshold)\n")

display(top_10)

print("\n" + "=" * 80)
print(f"Note: {(top_10['predicted_timestomped'] == 1).sum()} of these exceed the 0.5 classification threshold")
print("=" * 80)

TOP 10 HIGHEST CONFIDENCE PREDICTIONS

Showing top 10 events by confidence score:
(These are the most suspicious events regardless of classification threshold)



,eventtime,lf_lsn,usn_usn,filename,filepath,confidence_score,predicted_timestomped,classification,source,has_logfile_evidence,has_usnjrnl_evidence,cross_artifact_validation_score,timestamp_manipulation_pattern_score
16932,2023-01-05 22:09:00,6.281031e+09,996867144.0,boof.exe,\Windows\SysWOW64\boof.exe,0.508848,1,Timestomped,both,True,True,3,2
16931,2023-01-05 22:09:00,6.281030e+09,996866744.0,boof.dll,\Windows\SysWOW64\boof.dll,0.498856,0,Benign,both,True,True,3,2
17383,NaN,6.282905e+09,NaN,OneDrive.exe,\Program Files\Microsoft OneDrive\OneDrive.exe,0.459047,0,Benign,logfile_only,True,False,2,1
17416,NaN,6.282972e+09,NaN,OneDriveStandaloneUpdater.exe,\Program Files\Microsoft OneDrive\OneDriveStan...,0.439121,0,Benign,logfile_only,True,False,2,1
17380,NaN,6.281032e+09,NaN,boof.sys,\Windows\SysWOW64\boof.sys,0.429330,0,Benign,logfile_only,True,False,2,1
16954,2023-01-05 22:09:00,NaN,996866824.0,boof.dll,\Windows\SysWOW64\boof.dll,0.407020,0,Benign,usnjrnl_only,False,True,1,3
16992,2023-01-05 22:09:00,NaN,996867224.0,boof.exe,\Windows\SysWOW64\boof.exe,0.407020,0,Benign,usnjrnl_only,False,True,1,3
16994,2023-01-05 22:09:00,NaN,996867624.0,boof.sys,\Windows\SysWOW64\boof.sys,0.392331,0,Benign,usnjrnl_only,False,True,1,3
17414,NaN,6.282961e+09,NaN,OneDrive.VisualElementsManifest.xml,\Program Files\Microsoft OneDrive\OneDrive.Vis...,0.368998,0,Benign,logfile_only,True,False,2,1
16978,2023-01-05 22:09:00,6.281335e+09,997009840.0,BIT795C.tmp,\Windows\Temp\BIT795C.tmp,0.329558,0,Benign,both,True,True,3,2



Note: 1 of these exceed the 0.5 classification threshold


## 6. Save Results

In [41]:
print("=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Save 1: All predictions
print("\n1. Saving all predictions...")
predictions_file = OUTPUT_DIR / 'predictions.csv'
df.to_csv(predictions_file, index=False, encoding='utf-8-sig')
print(f"   Saved: {predictions_file}")
print(f"   Records: {len(df):,}")

# Save 2: Only flagged events (timestomped)
print("\n2. Saving flagged events...")
flagged = df[df['predicted_timestomped'] == 1].copy()

if len(flagged) > 0:
    # Sort by confidence (highest first)
    flagged = flagged.sort_values('confidence_score', ascending=False)
    
    # Select key columns for investigation including LSN/USN for traceability
    flagged_output_cols = [
        'eventtime', 'filename', 'filepath',
        'predicted_timestomped', 'confidence_score', 'confidence_level', 'classification',
        'source', 'has_logfile_evidence', 'has_usnjrnl_evidence',
        'lf_event', 'usn_event_info',
        'cross_artifact_validation_score',
        'timestamp_manipulation_pattern_score',
        'is_tunneling'
    ]
    
    # Add LSN/USN columns if they exist in the dataframe
    if 'lf_lsn' in df.columns:
        flagged_output_cols.insert(4, 'lf_lsn')
    if 'lf_cluster_index' in df.columns:
        flagged_output_cols.insert(5, 'lf_cluster_index')
    if 'usn_usn' in df.columns:
        flagged_output_cols.insert(6, 'usn_usn')
    
    # Filter to only columns that exist
    flagged_output_cols = [col for col in flagged_output_cols if col in flagged.columns]
    flagged_output = flagged[flagged_output_cols].copy()
    
    flagged_file = OUTPUT_DIR / 'flagged_files.csv'
    flagged_output.to_csv(flagged_file, index=False, encoding='utf-8-sig')
    print(f"   Saved: {flagged_file}")
    print(f"   Flagged events: {len(flagged):,}")
    
    # Show what traceability columns were included
    trace_cols = [col for col in ['lf_lsn', 'lf_cluster_index', 'usn_usn'] if col in flagged_output.columns]
    if trace_cols:
        print(f"   Traceability columns included: {', '.join(trace_cols)}")
else:
    print("   No timestomped events detected - no flagged file created")

print("\n" + "=" * 80)
print("Results saved successfully")
print("=" * 80)
print(f"\nOutput files:")
print(f"  - {predictions_file.name} (all events)")
if len(flagged) > 0:
    print(f"  - {flagged_file.name} (flagged events only)")
print(f"\nReady for analysis (Step 4)")

SAVING RESULTS

1. Saving all predictions...
   Saved: /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/12-KSK/predictions.csv
   Records: 17,418

2. Saving flagged events...
   Saved: /Users/soni/Github/Digital-Detectives_Thesis/data/prototype_output/12-KSK/flagged_files.csv
   Flagged events: 1
   Traceability columns included: lf_lsn, lf_cluster_index, usn_usn

Results saved successfully

Output files:
  - predictions.csv (all events)
  - flagged_files.csv (flagged events only)

Ready for analysis (Step 4)


## 7. Quick Preview of Flagged Events

In [42]:
if timestomped_count > 0:
    print("=" * 80)
    print("FLAGGED EVENTS PREVIEW")
    print("=" * 80)
    
    preview_cols = [
        'eventtime', 'filename', 'filepath',
        'confidence_score', 'source',
        'cross_artifact_validation_score'
    ]
    
    # Add LSN/USN if available
    if 'lf_lsn' in flagged.columns:
        preview_cols.insert(4, 'lf_lsn')
    if 'usn_usn' in flagged.columns:
        preview_cols.insert(5, 'usn_usn')
    
    # Filter to existing columns
    preview_cols = [col for col in preview_cols if col in flagged.columns]
    
    flagged_preview = flagged.head(10)[preview_cols]
    
    print(f"\nTop {min(10, len(flagged))} flagged events (by confidence):\n")
    display(flagged_preview)
    
    print("\n" + "=" * 80)
    print("Note: Run 04_View_Results.ipynb for detailed analysis")
    print("=" * 80)
else:
    print("=" * 80)
    print("NO TIMESTOMPED EVENTS DETECTED")
    print("=" * 80)
    print("\nAll events were classified as benign.")
    print("This could mean:")
    print("  1. No timestamp manipulation occurred in this case")
    print("  2. Only benign timestamp changes (e.g., file system tunneling)")
    print("  3. Manipulation patterns outside the model's training distribution")


FLAGGED EVENTS PREVIEW

Top 1 flagged events (by confidence):



,eventtime,filename,filepath,confidence_score,lf_lsn,usn_usn,source,cross_artifact_validation_score
16932,2023-01-05 22:09:00,boof.exe,\Windows\SysWOW64\boof.exe,0.508848,6.281031e+09,996867144.0,both,3



Note: Run 04_View_Results.ipynb for detailed analysis


---

## ✓ Step 3 Complete!

**Next:** Run `04_View_Results.ipynb` to analyze flagged events in detail.


---